# DataMate v1 — Kaggle runner
Attach the **source package**, **DataMate data**, **complete Qwen3.5-9B checkpoint**, and an optional **compatible wheelhouse**. Select a GPU. Check the discovered paths in Cell 1. This notebook executes local code and local weights; it does not call an external model API.

Run cells in order. Start with a small training subset. The full run is enabled explicitly in Cell 10. A GPU smoke test is required; 4-bit Qwen3.5 compatibility is hardware/version dependent. Read `README.md` for environment preparation and troubleshooting.


## 1. Attached paths

In [ ]:
# Cell 1 — automatically discover attached inputs

from pathlib import Path
import hashlib
import json
import os
import sys
import zipfile

INPUT_ROOT = Path("/kaggle/input")
WORK = Path("/kaggle/working/datamate")
WORK.mkdir(parents=True, exist_ok=True)

# Choose "train" or "public_test".
SPLIT_NAME = "train"

print("Attached Kaggle inputs:")

for path in sorted(INPUT_ROOT.iterdir()):
    print(" -", path)


# ---------------------------------------------------------
# Locate the DataMate source
# ---------------------------------------------------------

REQUIRED_SOURCE_VERSION = '1.0.4'


def compatible_source(read_source):
    try:
        return (
            f'__version__ = "{REQUIRED_SOURCE_VERSION}"' in read_source("datamate/__init__.py")
            and "prefill_chunk_size:" in read_source("datamate/config.py")
            and "prefill_chunk_size=chunk_size" in read_source("datamate/llm.py")
            and "prefill_chunk_size:" in read_source("configs/kaggle.yaml")
        )
    except (FileNotFoundError, KeyError):
        return False


source_candidates = sorted(path.parent.parent for path in INPUT_ROOT.rglob("datamate/cli.py"))
SOURCE = None
for candidate in source_candidates:
    if compatible_source(lambda name: (candidate / name).read_text(encoding="utf-8")):
        SOURCE = candidate
        break
    print("Skipping incompatible source folder:", candidate)

if SOURCE is None:
    for archive in sorted(INPUT_ROOT.rglob("Agent_v1_source.zip")):
        with zipfile.ZipFile(archive) as zip_file:
            if not compatible_source(lambda name: zip_file.read("Agent_v1/" + name).decode("utf-8")):
                print("Skipping incompatible source archive:", archive)
                continue
            archive_hash = hashlib.sha256(archive.read_bytes()).hexdigest()
            destination = (WORK / "source" / archive_hash).resolve()
            for member in zip_file.infolist():
                if not (destination / member.filename).resolve().is_relative_to(destination):
                    raise ValueError(f"Unsafe ZIP member: {member.filename}")
            destination.mkdir(parents=True, exist_ok=True)
            zip_file.extractall(destination)
            SOURCE = destination / "Agent_v1"
            print("Extracted compatible source:", archive)
            break

if SOURCE is None:
    raise FileNotFoundError(
        "No compatible DataMate source found. Attach the updated Agent_v1_source.zip "
        "or replace the extracted Agent_v1 folder with a complete copy."
    )


# ---------------------------------------------------------
# Locate the challenge split
# ---------------------------------------------------------

split_candidates = []

for questions_path in INPUT_ROOT.rglob(
    "questions.jsonl"
):
    candidate = questions_path.parent

    if (
        candidate.name == SPLIT_NAME
        and (candidate / "data").is_dir()
        and (candidate / "rulebook.md").is_file()
        and (candidate / "calendar.csv").is_file()
    ):
        split_candidates.append(candidate)

split_candidates = sorted(set(split_candidates))

if not split_candidates:
    raise FileNotFoundError(
        f"Could not find a complete {SPLIT_NAME!r} "
        "split under /kaggle/input. Attach the "
        "DataMate challenge dataset."
    )

if len(split_candidates) > 1:
    print("Multiple data split candidates found:")

    for candidate in split_candidates:
        print(" -", candidate)

SPLIT = split_candidates[0]


# ---------------------------------------------------------
# Locate Qwen3.5-9B
# ---------------------------------------------------------

model_candidates = []

for config_path in INPUT_ROOT.rglob("config.json"):
    try:
        config_data = json.loads(
            config_path.read_text(
                encoding="utf-8"
            )
        )

        architectures = str(
            config_data.get("architectures", [])
        )

        model_type = str(
            config_data.get("model_type", "")
        )

        if (
            "Qwen3_5" in architectures
            or model_type == "qwen3_5"
        ):
            candidate = config_path.parent

            if (
                candidate
                / "model.safetensors.index.json"
            ).is_file():
                model_candidates.append(candidate)

    except Exception:
        pass

model_candidates = sorted(set(model_candidates))

if not model_candidates:
    raise FileNotFoundError(
        "Could not find the attached Qwen3.5-9B "
        "checkpoint. Attach the saved output of the "
        "download-qwen35-9b notebook."
    )

if len(model_candidates) > 1:
    print("Multiple Qwen candidates found:")

    for candidate in model_candidates:
        print(" -", candidate)

MODEL = model_candidates[0]


# ---------------------------------------------------------
# Optional wheelhouse
# ---------------------------------------------------------

wheel_candidates = sorted({
    path
    for path in INPUT_ROOT.rglob("*")
    if path.is_dir()
    and "wheel" in path.name.lower()
})

WHEELS = (
    wheel_candidates[0]
    if wheel_candidates
    else Path("/kaggle/input/datamate-wheels")
)


# ---------------------------------------------------------
# Final validation
# ---------------------------------------------------------

assert (
    SOURCE / "datamate" / "cli.py"
).is_file()

assert (
    SOURCE / "configs" / "kaggle.yaml"
).is_file()

assert (
    SPLIT / "questions.jsonl"
).is_file()

assert (
    SPLIT / "rulebook.md"
).is_file()

assert (
    SPLIT / "calendar.csv"
).is_file()

assert (
    MODEL / "config.json"
).is_file()

assert (
    MODEL / "model.safetensors.index.json"
).is_file()

refreshed_source = None
sys.path.insert(0, str(SOURCE))

os.environ["PYTHONPATH"] = (
    str(SOURCE)
    + os.pathsep
    + os.environ.get("PYTHONPATH", "")
)

print("\nResolved paths:")
print("SOURCE =", SOURCE)
print("SPLIT  =", SPLIT)
print("MODEL  =", MODEL)
print("WHEELS =", WHEELS)
print("WORK   =", WORK)

## 2. Hardware inspection (before dependency changes)

In [ ]:
import platform, shutil, subprocess, importlib.util
print("Python", platform.python_version(), "Free disk GiB", shutil.disk_usage(WORK).free / 2**30)
subprocess.run(["nvidia-smi"], check=False)
if importlib.util.find_spec("torch"):
    import torch
    print("Torch", torch.__version__, "CUDA", torch.version.cuda)
    for i in range(torch.cuda.device_count()):
        print(i, torch.cuda.get_device_name(i), torch.cuda.get_device_capability(i), torch.cuda.get_device_properties(i).total_memory / 2**30)


## 3. Dependencies
Use DEPENDENCY_SOURCE='wheelhouse' with an attached compatible wheelhouse. Use DEPENDENCY_SOURCE='internet' for an Internet-enabled setup session. Start with a fresh kernel so the package versions used by the notebook match its SQL workers.

In [ ]:
# Cell 3 — install dependencies automatically

from pathlib import Path
import importlib
import importlib.metadata
import importlib.util
import subprocess
import sys

# Choose one:
DEPENDENCY_SOURCE = "internet"
# DEPENDENCY_SOURCE = "wheelhouse"

requirements = SOURCE / "requirements-kaggle.txt"

assert requirements.is_file(), requirements

command = [
    sys.executable,
    "-m",
    "pip",
    "install",
    "--disable-pip-version-check",
    "--no-input",
]

if DEPENDENCY_SOURCE == "wheelhouse":
    assert WHEELS.is_dir(), (
        f"Wheelhouse was not found: {WHEELS}"
    )

    command += [
        "--no-index",
        "--find-links",
        str(WHEELS),
    ]

command += [
    "-r",
    str(requirements),
]

print("Installing dependencies from:", DEPENDENCY_SOURCE)
print("Requirements:", requirements)

subprocess.check_call(command)

importlib.invalidate_caches()

required_modules = [
    "duckdb",
    "sqlglot",
    "pydantic",
    "yaml",
    "openpyxl",
    "simplejson",
    "pandas",
    "torch",
    "transformers",
    "accelerate",
    "bitsandbytes",
]

missing = [
    module
    for module in required_modules
    if importlib.util.find_spec(module) is None
]

assert not missing, f"Missing dependencies: {missing}"

print("\nDependency installation completed.")

for package in [
    "duckdb",
    "sqlglot",
    "pydantic",
    "PyYAML",
    "openpyxl",
    "simplejson",
    "pandas",
    "torch",
    "transformers",
    "accelerate",
    "bitsandbytes",
]:
    try:
        version = importlib.metadata.version(package)
        print(f"{package}: {version}")
    except importlib.metadata.PackageNotFoundError:
        print(f"{package}: version unavailable")

## 4. Offline configuration
The Kaggle NF4/FP16 profile processes long prompts in 512-token chunks and retains the full prompt.

In [ ]:
for key in ["HF_HUB_OFFLINE", "TRANSFORMERS_OFFLINE", "HF_DATASETS_OFFLINE"]:
    os.environ[key] = "1"
import logging
logging.basicConfig(level=logging.INFO, format="%(levelname)s %(message)s")
logging.getLogger("datamate").setLevel(logging.INFO)
from importlib import import_module, invalidate_caches
import shutil
import tempfile

required_source_version = "1.0.4"
expected_config = (SOURCE / "datamate/config.py").resolve()
version_file = SOURCE / "datamate/__init__.py"
if (
    f'__version__ = "{required_source_version}"' not in version_file.read_text(encoding="utf-8")
    or "prefill_chunk_size:" not in expected_config.read_text(encoding="utf-8")
):
    raise RuntimeError(
        f"The file on disk at {expected_config} is from an older DataMate source. "
        "Attach the complete updated Agent_v1_source.zip and rerun Cell 1."
    )

import_root = SOURCE
if globals().get("refreshed_source") == SOURCE:
    import_root = refreshed_import_root

datamate_config = import_module("datamate.config")
loaded_config = Path(datamate_config.__file__).resolve()
if loaded_config != (import_root / "datamate/config.py").resolve() or (
    getattr(import_module("datamate"), "__version__", None) != required_source_version
    or "prefill_chunk_size" not in datamate_config.ModelConfig.model_fields
):
    # Import a clean copy of the selected package, including its sibling modules.
    # A new directory avoids reusing cached bytecode from a mounted dataset.
    import_root = Path(tempfile.mkdtemp(prefix="datamate-import-", dir=WORK))
    shutil.copytree(SOURCE / "datamate", import_root / "datamate",
                    ignore=shutil.ignore_patterns("__pycache__", "*.pyc"))
    globals().pop("agent", None)
    for name in tuple(sys.modules):
        if name == "datamate" or name.startswith("datamate."):
            del sys.modules[name]
    sys.path.insert(0, str(import_root))
    invalidate_caches()
    datamate_config = import_module("datamate.config")
    loaded_config = Path(datamate_config.__file__).resolve()
    if loaded_config != (import_root / "datamate/config.py").resolve() or (
        getattr(import_module("datamate"), "__version__", None) != required_source_version
        or "prefill_chunk_size" not in datamate_config.ModelConfig.model_fields
    ):
        raise RuntimeError(f"Could not import the updated DataMate package from {import_root}.")
    refreshed_source, refreshed_import_root = SOURCE, import_root
    print("Refreshed DataMate imports from the selected source.")

worker_paths = os.environ.get("PYTHONPATH", "").split(os.pathsep)
os.environ["PYTHONPATH"] = os.pathsep.join(
    [str(import_root), *(path for path in worker_paths if path and path != str(import_root))]
)
print("DataMate config loaded from:", loaded_config)
from datamate.runtime import environment
from datamate.util import write_json
cfg = datamate_config.Config.load(SOURCE / "configs/kaggle.yaml", split=SPLIT, model=MODEL, work=WORK)
# Example for a supported >=32GB GPU: cfg.model.quantization="none"; cfg.model.dtype="bfloat16"
# For a one-GPU NF4 run, cfg.model.device_map="cuda:0" can avoid unintended CPU offload.
cfg.runtime.fail_fast = True  # surface programming/environment problems during development
cfg.validate_paths()
write_json(WORK / "environment.json", environment(WORK))
print(cfg.model_dump())


## 5. Load once and smoke-test
This fingerprints local weights, loads Qwen, and checks structured generation. Stop here if it fails. No random-weight fallback or automatic model download is permitted.

In [ ]:
from datamate.pipeline import Agent, read_questions
agent = Agent(cfg)
print("Backend ready:", agent.backend.identity)


## 6. Select a development business
IDs here only select input data; the agent has no identifier-specific answer logic.

In [ ]:
questions = read_questions(SPLIT)
DEV_BUSINESSES = [questions[0].business_id]  # Set to None for split-wide selection.
DEV_OFFSET = 0  # Set to 20 to start at question 21.
DEV_LIMIT = 20
selected = read_questions(SPLIT, DEV_BUSINESSES, DEV_LIMIT, offset=DEV_OFFSET)
print("Total", len(questions), "Development", len(selected), "Business", DEV_BUSINESSES)


## 7. Inspect prepared schema and parsing diagnostics

In [ ]:
context = agent.prepare(selected[0].business_id)
for table in context["catalog"]:
    print(table["table"], table["role"], table["grain"], "rows=", table["row_count"],
          "duplicate_rows=", table.get("duplicate_rows", 0), "parse_errors=", table["parse_errors"])
print("Database:", context["database"])


## 8. End-to-end development inference

In [ ]:
def diagnose(run):
    subprocess.run([sys.executable, "-X", "utf8", str(SOURCE / "scripts/diagnose_run.py"), str(run)], check=True)

dev_run = agent.predict(DEV_BUSINESSES, DEV_LIMIT, offset=DEV_OFFSET)
print("Development run:", dev_run)
diagnose(dev_run)


## 9. Training-only evaluation
Reference answers are read only by this scorer, never by the predictor. selected_only=True scores the development subset; full competition scores use all questions.

In [ ]:
from datamate.evaluation import evaluate, read_jsonl
if (SPLIT / "answers.jsonl").is_file():
    metrics = evaluate(dev_run / "submission.jsonl", SPLIT / "answers.jsonl", SPLIT / "questions.jsonl", dev_run / "metrics.json", selected_only=True)
    print(metrics)
    references = read_jsonl(SPLIT / "answers.jsonl")
    for qid, record in read_jsonl(dev_run / "submission.jsonl").items():
        expected = references[qid]
        print(qid, "predicted", record["type"], record.get("answer"),
              "expected", expected["type"], expected.get("answer"))
else:
    print("No released references for this split; inference only.")


## 10. Full prediction / resume
Enable after inspecting the development logs. The current Agent's deadline includes its loading and prior work. For a fresh timed run, restart and load once. To resume after a session interruption, restore saved work outputs and use identical source, input files, config, backend and model. Prior partial answers are fingerprint-checked.

In [ ]:
RUN_FULL = False
if RUN_FULL:
    cfg.runtime.fail_fast = False
    full_run = agent.predict()
    print("Full run:", full_run)
    diagnose(full_run)
else:
    print("Set RUN_FULL=True to process all questions.")


## 11. Validate and save outputs
Download/save the submission, run logs, configuration and manifest as Kaggle notebook outputs. A development-only submission intentionally lacks the other IDs and is not a full submission.

In [ ]:
from datamate.evaluation import read_jsonl
from datamate.output import validate_record
submission = WORK / "submission.jsonl"
records = read_jsonl(submission)
for record in records.values():
    validate_record(record)
expected = {q.id for q in questions}
print("Records:", len(records), "Missing IDs:", len(expected - records.keys()), "Extra IDs:", len(records.keys() - expected))
print("Submission:", submission)
print("Run artifacts:", WORK / "runs")
if RUN_FULL:
    assert set(records) == expected
